## Imports

In [ ]:
import numpy as np
import xarray as xr
import copy
import src.utils
import src.lme_utils
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cmocean
import os
import pathlib
import cartopy.crs as ccrs
import pandas as pd
import scipy.stats
import tqdm
import time
import xeofs as xe

## Set seaborn plotting style
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## initialize RNG
rng = np.random.default_rng(seed=100)

## Funcs

In [ ]:
## specify lons/lats for E/W boxes
LONS_W = [120, 200]
LONS_E = [200, 280]


## funcs to plot boxes
def plot_wbox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_W, lats=[-5, 5], **kwargs)


def plot_ebox(ax, **kwargs):
    src.utils.plot_box(ax, lons=LONS_E, lats=[-5, 5], **kwargs)


## funcs to average over regions
def get_eq_avg(data, lons):
    """average over equatorial region and longitudes"""
    return data.sel(longitude=slice(*lons), latitude=slice(-5, 5)).mean(
        ["longitude", "latitude"]
    )


def get_e(data):
    return get_eq_avg(data, LONS_E)


def get_w(data):
    return get_eq_avg(data, LONS_W)


def get_dx(data):
    return get_e(data) - get_w(data)


def add_gridlines(axs):
    """func to add gridlines to axs object"""

    for ax in axs[:-1, 0]:
        ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[],
            ylocs=[-30, 0, 30],
        )
    for ax in axs[-1]:
        gl_ = ax.gridlines(
            crs=ccrs.PlateCarree(),
            draw_labels=True,
            linewidth=1,
            alpha=0,
            xlocs=[50, 120 - 160],
            # xlocs=[],
            ylocs=[-30, 0, 30],
        )
        gl_.top_labels = False
        # gl_.bottom_labels = True
    # gl_.left_labels = False

    return


def plot_level(ax, data, level, ls="-", c="white"):
    """plot single level on hovmoller"""
    ax.contour(
        data.longitude,
        data.latitude,
        data,
        levels=[level],
        colors=c,
        linestyles=ls,
        zorder=10,
        transform=ccrs.PlateCarree(),
    )
    return


import cartopy.crs as ccrs


def make_cb_range(dx, nlevs):
    amp = dx * nlevs - dx / 2

    lev1 = np.arange(0, amp + dx / 2, dx) + dx / 2
    lev0 = -lev1[::-1]
    return np.concatenate([lev0, lev1])


def plot_pr_diff(ax, data, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.balance_r",
        levels=make_cb_range(dx, nlev),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_sst_sigma(ax, data, lev_min=0.3, dx=0.2, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.amp",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp


def plot_pr(ax, data, lev_min=1, dx=1, nlev=5):
    """plot data on ax"""

    cp = ax.contourf(
        data.longitude,
        data.latitude,
        data,
        cmap="cmo.rain",
        levels=np.arange(lev_min, lev_min + dx * (nlev + 1), dx),
        transform=ccrs.PlateCarree(),
        extend="both",
    )

    return cp

## Data loading

In [ ]:
def sel_area(data, lon_range=None, lat_range=None):
    """select area on t grids"""

    ## find in lon range
    if lon_range is None:
        lon_idx = np.array([True for _ in data.nlon])
    else:
        lon_idx = (
            ((data.TLONG >= lon_range[0]) & (data.TLONG <= lon_range[1]))
            .any("nlat")
            .values
        )

    if lat_range is None:
        lat_idx = np.array([True for _ in data.nlat])
    else:
        lat_idx = (
            ((data.TLAT >= lat_range[0]) & (data.TLAT <= lat_range[1]))
            .any("nlon")
            .values
        )

    return data.isel(nlon=lon_idx, nlat=lat_idx)


def area_avg(data, varname, lon_range=None, lat_range=None):
    """average over area"""

    ## trim data
    data_ = sel_area(data, lon_range=lon_range, lat_range=lat_range)

    ## get dims to sum over
    integrate = lambda x: (x * data_["dA"]).sum(["nlon", "nlat"])

    return integrate(data_[varname]) / integrate(1.0)


def get_eli_helper(exceeds_thresh):
    """compute ELI from mask of threshold exceedance"""

    ## sum and count longitudes exceeding thresh
    lon = exceeds_thresh.longitude
    longitude_sum = (exceeds_thresh * lon).sum(["longitude", "latitude"])
    longitude_count = exceeds_thresh.sum(["longitude", "latitude"])

    ## eli is average longitude
    eli = longitude_sum / longitude_count

    return eli


def get_eli(rsst, rsst_thresh=0, max_lat=15):
    """compute ELI from tropical SST data"""

    ## get equatorial Pac. SST
    rsst_pac = rsst.sel(longitude=slice(120, 280), latitude=slice(-max_lat, max_lat))

    ## get boolean array where SST exceeds thresh
    exceeds_thresh0 = rsst_pac >= rsst_thresh

    ## compute initial ELI estimate
    eli0 = get_eli_helper(exceeds_thresh0)

    return eli0

## Load the data

### init. cluster

In [ ]:
from dask.distributed import LocalCluster, Client

cluster = LocalCluster(n_workers=2)
client = Client(cluster)
client

### Do loading

In [ ]:
SAVE_FP = pathlib.Path("/glade/work/kcarr/lme_data")
X = xr.open_dataset(SAVE_FP / "snr" / "pr_split.nc")

## get anomalies and trim to WPAC
X = X["anom"].sel(lon=slice(60, 140), time=slice(None, "1850-01"))

## fix time coord
time_fixed = xr.date_range(start="0850-01", end="1849-12", freq="MS")
X = X.assign_coords({"time": time_fixed})

## weight data sst by coslat
W = src.utils.get_coslat_weights(X)
X = X * W

## LFCA compute

user args

In [ ]:
## truncate to NEOFS
NEOFS = 30

## specify dims
T_DIM = "time"
DIMS = [T_DIM, "member"]

### Compute eofs on original data

In [ ]:
## specs for EOFs
eofs_kwargs = dict(n_modes=NEOFS, standardize=False, use_coslat=False, center=True)

## initialize EOF model
eofs = xe.single.EOF(**eofs_kwargs)

## compute
eofs.fit(X, dim=["time", "member"])

## extract components
N = int(len(X.member) * len(X.time))
Q = eofs.scores(normalized=True)
Z = eofs.components(normalized=True)
sigma = np.sqrt(N * eofs.explained_variance())

## merge data
eofs_ = xr.merge(
    [
        Z.rename("Z"),
        sigma.rename("sigma"),
        Q.rename("Q"),
        W.rename("W"),
    ]
)

Test things make sense

In [ ]:
## compute inner products
n_testmode = 10
idx = dict(mode=slice(None, n_testmode))
QtQ = xr.dot(
    eofs_["Q"].isel(idx),
    eofs_["Q"].isel(idx).rename({"mode": "mode_out"}),
    dims=DIMS,
)

ZtZ = xr.dot(
    eofs_["Z"].isel(idx).fillna(0),
    eofs_["Z"].isel(idx).fillna(0).rename({"mode": "mode_out"}),
    dims=["lat", "lon"],
)

## check they're identities
eye = np.eye(n_testmode)
print(np.allclose(ZtZ.values, eye))
print(np.allclose(QtQ.values, eye))

### Now do LFCA part

In [ ]:
from scipy.signal import butter, lfilter, freqz, filtfilt


def butter_lowpass(cutoff, fs, order=5):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return b, a


def butter_lowpass_filter(data, cutoff, fs, order=5, axis=-1):
    b, a = butter_lowpass(cutoff, fs, order=order)
    y = filtfilt(b, a, data, axis=axis)
    return y.squeeze()

In [ ]:
## do low-pass filtering
Qtilde = copy.deepcopy((eofs_["Q"]).transpose(..., DIMS[0]))

fs = 12 if "time" in DIMS else 1

Qtilde.values = butter_lowpass_filter(
    Qtilde.values[None, :], cutoff=1 / 25, fs=fs, axis=-1
)
eofs_["Qtilde"] = Qtilde

In [ ]:
## plot results
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(eofs_["Q"].isel(mode=0, member=0).isel({T_DIM: slice(None, 500 * 12)}))
ax.plot(eofs_["Qtilde"].isel(mode=0, member=0).isel({T_DIM: slice(None, 500 * 12)}))

plt.show()

In [ ]:
## covariance
N = int(len(eofs_.stack(s=DIMS).s))
QtQ = 1 / N * (eofs_["Qtilde"] * eofs_["Qtilde"].rename({"mode": "mode_"})).sum(DIMS)

## eigendecomp
Lambda_sqr, U = np.linalg.eigh(QtQ.values)
U = U[:, ::-1]
Lambda_sqr = Lambda_sqr[::-1]

## put in xarray dataset
eofs_tilde = xr.Dataset(
    data_vars=dict(
        U=(("mode", "mode_tilde"), U),
        Lambda=("mode_tilde", np.sqrt(Lambda_sqr)),
    ),
    coords=dict(mode=eofs_.mode.values, mode_tilde=eofs_.mode.values),
)

## get timeseries timeseries
eofs_tilde["tk"] = xr.dot(eofs_tilde["U"], eofs_["Q"], dim="mode")
eofs_tilde["tk_tilde"] = xr.dot(eofs_tilde["U"], eofs_["Qtilde"], dim="mode")

# get patterns
eofs_tilde["P"] = xr.dot(
    eofs_["Z"].fillna(0),
    eofs_["sigma"],
    eofs_tilde["U"],
    dim="mode",
)

## check tk's are orthgonal
TtT = xr.dot(
    eofs_tilde["tk"],
    eofs_tilde["tk"].rename({"mode_tilde": "mode"}),
    dim=DIMS,
)

## do tests
# print(np.allclose(eofs_tilde["P"], eofs_tilde["P_test"]))
print(np.allclose(TtT.values, np.eye(TtT.shape[0])))

### compute full spatial patterns

In [ ]:
def get_P(eof_data, tk, dims):
    """get spatial pattern from projected data"""

    ## first, get projected pattern
    P_proj = xr.dot(eof_data["Q"], tk, dim=dims)

    ## next, reconstruct spatial pattern
    P = xr.dot(eof_data["Z"], eof_data["sigma"], P_proj, dim="mode")

    return P

In [ ]:
## load other data to compute patterns
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)

## load precip
pr_full = xr.open_dataset(
    pathlib.Path(SAVE_FP, "snr", "eofs_pr_deseason.nc"), decode_times=time_coder
)
pr_full = pr_full.sel(time=slice(None, "1850-01")).assign_coords({"time": time_fixed})

## load sst
sst_full = xr.open_dataset(
    pathlib.Path(SAVE_FP, "snr", "eofs_sst_trop_deseason.nc"), decode_times=time_coder
)
sst_full = sst_full.sel(time=slice(None, "1849-12"))

In [ ]:
## compute patterns for SST and precip
P_sst = get_P(eof_data=sst_full, tk=eofs_tilde["tk"], dims=DIMS)
P_pr = get_P(eof_data=pr_full, tk=eofs_tilde["tk"], dims=DIMS)

#### Plot structure of EOFs

In [ ]:
fig = plt.figure(figsize=(10, 7), layout="constrained")
format_func = lambda ax,: src.utils.plot_setup_pac(ax, max_lat=27, lon_range=(60, 280))
axs = src.utils.subplots_with_proj(fig, nrows=3, ncols=2, format_func=format_func)


for j in range(3):
    for i, (P_, dx) in enumerate([(P_sst, 4e1), (-P_pr, 1.5e2)]):
        cp00 = plot_pr_diff(
            axs[j, i],
            P_.isel(mode_tilde=j).rename({"lat": "latitude", "lon": "longitude"}),
            dx=dx,
        )

    ## label
    axs[j, 0].set_title(f"Mode {j}", loc="left")

## add labels
# add_gridlines(axs)
bbox = dict(boxstyle="round", facecolor="white", alpha=1)
legend_kwargs = dict(x=0.01, y=0.02, fontdict=dict(size=12, color="gray"), bbox=bbox)
for ax in axs.flatten():
    ax.set_aspect("auto")
    src.utils.plot_box(ax, lons=[65, 150], lats=[-25, 25], c="k", ls="--")

## colorbars
cb_kwargs = dict(label=r"$^{\circ}$C", pad=0.03)
# fig.colorbar(cp00, ax=axs[0,0], ticks=[-3.6,0,3.6], **cb_kwargs)
# fig.colorbar(cp10, ax=axs[1,0], ticks=[-1.8,0,1.8], **cb_kwargs)
# add_gridlines(axs)
plt.show()

#### Spectrum

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.scatter(eofs_tilde.mode_tilde, eofs_tilde.Lambda)
ax.set_ylim([0, None])

#### Timeseries

In [ ]:
sel = lambda x: x.isel(member=0, mode_tilde=0).isel({T_DIM: slice(None, None)})
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(-sel(eofs_tilde["tk"]))
ax.plot(-sel(eofs_tilde["tk_tilde"]))
plt.show()

#### Scatter

To-do: scatter vs. $\sigma(T)$ as well!

In [ ]:
## specify plot modes
mode0, mode1 = 0, 1

fig, axs = plt.subplots(1, 2, figsize=(7, 3.5), layout="constrained")

for ax, v, title in zip(axs, ["tk", "tk_tilde"], ["raw", "25-yr low-pass"]):

    ## get appropriate data
    tk = eofs_tilde[v].isel(time=slice(600, None))

    ## extract
    x0, x1 = tk.isel(mode_tilde=mode0), tk.isel(mode_tilde=mode1)

    ## plot
    ax.scatter(x0, x1, s=1)

    ## label
    corr = xr.corr(x0, x1).values.item()
    ax.set_title(f"{title}, $r=${corr:.2f}")

    ## format
    ax_kwargs = dict(ls="--", c="gray", lw=1)
    ax.axhline(**ax_kwargs)
    ax.axvline(**ax_kwargs)
    ax.set_xlabel(f"Mode {mode0}")


axs[0].set_ylabel(f"Mode {mode1}")
# src.utils.set_lims(axs)
plt.show()

Compute $\sigma(T)$

In [ ]:
def recon_fn(eof_data, fn=lambda x: x):
    """reconstruct given function from eof data"""

    return xr.dot(fn(eof_data["Z"]), eof_data["sigma"], eof_data["Q"], dim=["mode"])

In [ ]:
## reconstruct Niño 3
T3 = recon_fn(
    sst_full.rename({"lat": "latitude", "lon": "longitude"}), fn=src.utils.get_nino3
)

In [ ]:
nyears = 25
nmonths = nyears * 12
nmonths_2 = int(nmonths / 2)
T3_var = T3.rolling({"time": nmonths + 1}, center=True).var("time")
T3_var = T3_var.isel(time=slice(nmonths_2, -nmonths_2))

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3.5), layout="constrained")


## extract
tk = eofs_tilde["tk_tilde"].isel(mode_tilde=0)
tk = tk.sel(time=T3_var.time)

## plot
ax.scatter(tk, T3_var, s=1)

## label
corr = xr.corr(tk, T3_var).values.item()
ax.set_title(r"Mode 0 vs. $\sigma^2(T_3)$, " + f"$r=${corr:.2f}")

## format
ax_kwargs = dict(ls="--", c="gray", lw=1)
ax.axhline(**ax_kwargs)
ax.axvline(**ax_kwargs)
ax.set_xlabel("Mode 0 (25-yr lowpass)")
ax.set_ylabel(r"$\sigma^2(T_3)$")

plt.show()